In [61]:
import math
import pandas as pd
import numpy as np
import random
import re
import string
from collections import defaultdict
from collections import Counter

In [62]:
splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}

base = "hf://datasets/cornell-movie-review-data/rotten_tomatoes/"
train_df = pd.read_parquet(base + splits['train'])
val_df = pd.read_parquet(base + splits['validation'])
test_df = pd.read_parquet(base + splits['test'])

# Preprocess

In [63]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower().translate(str.maketrans('', '', string.punctuation)))

In [64]:
tokens = [tokenize(t) for t in train_df["text"]]

In [65]:
flat_tokens = [w for sent in tokens for w in sent]
word_counts = Counter(flat_tokens)
vocab = sorted(word_counts.keys())
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
vocab_size = len(vocab)

# Train word2vec

In [66]:
EMBEDDING_SIZE = 100
CONTEXT_WINDOW_SIZE = 2
NEGATIVES_PER_POSITIVE = 2

In [67]:
def generate_skipgram_pairs(k):
    pairs = set()
    word_context_map = defaultdict(set)
    for sentence in tokens:
        ids = [word_to_id[w] for w in sentence if w in word_to_id]
        for i, center in enumerate(ids):
            # context window
            start = max(0, i - k)
            end = min(len(ids), i + k + 1)
            for j in range(start, end):
                if i == j:
                    continue
                pairs.add((center, ids[j]))
                word_context_map[center].add(ids[j])
    return pairs, word_context_map

In [68]:
pairs, word_context_map = generate_skipgram_pairs(k=CONTEXT_WINDOW_SIZE)

Exception ignored in: <function tqdm.__del__ at 0x11f4e89a0>
Traceback (most recent call last):
  File "/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/tqdm/std.py", line 1277, in close
    if self.last_print_t < self.start_t + self.delay:
       ^^^^^^^^^^^^^^^^^
AttributeError: 'tqdm_asyncio' object has no attribute 'last_print_t'


In [69]:
def sample_negative_words(word_idx, n):
    possible_negative_words = set(range(vocab_size)) - word_context_map[word_idx]
    negative_to_select = min(n, len(possible_negative_words))
    return random.sample(sorted(possible_negative_words), negative_to_select)

In [70]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [71]:
rng = np.random.default_rng(42)
V_in = rng.uniform(-0.5, 0.5, (vocab_size, EMBEDDING_SIZE))  # input embeddings
U_out = rng.uniform(-0.5, 0.5, (vocab_size, EMBEDDING_SIZE)) # output embeddings

In [72]:
def train_skipgram_vectorized_step(center_id, context_ids: list, lr):
    v_c = V_in[center_id]

    # positive
    U_pos_vec = U_out[context_ids]
    g_pos_vec = (1 - sigmoid(np.dot(U_pos_vec, v_c)))[:, np.newaxis] / len(context_ids)
    grad_v_pos_vec = np.sum(g_pos_vec * U_pos_vec, axis=0)
    U_out[context_ids] += lr * g_pos_vec * v_c[np.newaxis, :]

    # negative
    n_negatives = math.ceil(len(context_ids) * NEGATIVES_PER_POSITIVE)
    neg_ids = list(sample_negative_words(center_id, n_negatives))

    U_neg_vec = U_out[neg_ids]
    g_neg_vec = (-sigmoid(np.dot(U_neg_vec, v_c)))[:, np.newaxis] / len(neg_ids)
    grad_v_neg_vec = np.sum(g_neg_vec * U_neg_vec, axis=0)
    U_out[neg_ids] += lr * g_neg_vec * v_c[np.newaxis, :]

    V_in[center_id] += lr * (grad_v_pos_vec + grad_v_neg_vec)

In [73]:
V_history = []

In [74]:
def train_skipgram_vectorized(lr=0.1, epochs=1):
    for epoch in range(epochs):
        V_history.append(V_in.copy())

        for center, context in word_context_map.items():
            train_skipgram_vectorized_step(center, list(context), lr)
        print(f"Epoch {epoch+1}/{epochs} done")

In [ ]:
train_skipgram_vectorized(epochs=50, lr=0.03)

Epoch 1/50 done
Epoch 2/50 done
Epoch 3/50 done
Epoch 4/50 done
Epoch 5/50 done
Epoch 6/50 done
Epoch 7/50 done
Epoch 8/50 done
Epoch 9/50 done
Epoch 10/50 done
Epoch 11/50 done
Epoch 12/50 done
Epoch 13/50 done
Epoch 14/50 done


In [ ]:
np.save("V_in.npy", V_in)
np.save("U_out.npy", U_out)

In [40]:
V_in = np.load("V_in.npy")
U_out = np.load("U_out.npy")

array([ 0.28997852, -0.10410881,  0.43084449,  0.26736146, -0.36279785,
        0.50173143,  0.33288471,  0.34213077, -0.26480027,  0.07378583,
       -0.15062925,  0.35019311,  0.1022391 ,  0.3419177 , -0.03927989,
       -0.2294038 ,  0.07821908, -0.32731848,  0.31614874,  0.06643443,
        0.30315182, -0.13337603,  0.4680576 ,  0.38649152,  0.21880838,
       -0.22999207,  0.10100116, -0.44499107, -0.41369256,  0.18802512,
        0.2958272 ,  0.41988954, -0.19002777, -0.12356157, -0.07974818,
       -0.37283973, -0.29327701, -0.02416951, -0.26789813,  0.16598854,
        0.01415965,  0.41777213,  0.23984426, -0.04423751,  0.3186485 ,
        0.28008921, -0.11763883, -0.20657126,  0.22925005, -0.38016316,
       -0.27409358, -0.51247941,  0.26544282,  0.20133653,  0.30107506,
        0.29436453, -0.07241675,  0.10064341, -0.37778363, -0.39476481,
        0.16249767, -0.01338767, -0.0802744 ,  0.21917646,  0.08865982,
        0.04351726,  0.10863292, -0.15727908, -0.49437684, -0.05

# Sentiment CNN

In [41]:
import os
from tqdm.auto import tqdm
import math
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn

## Prepare vectors for CNN

In [51]:
# Add special PAD and UNK vectors
PAD = "<PAD>"
UNK = "<UNK>"
pad_id = len(word_to_id)
unk_id = pad_id + 1
word_to_id[PAD] = pad_id
word_to_id[UNK] = unk_id

rng = np.random.RandomState(42)
pad_vec = np.zeros((1, EMBEDDING_SIZE), dtype=np.float32)
unk_vec = rng.normal(scale=0.01, size=(1, EMBEDDING_SIZE)).astype(np.float32)
V_in_extended = np.vstack([V_in.astype(np.float32), pad_vec, unk_vec])

In [52]:
class SentimentDataset(Dataset):
    def __init__(self, df, word_to_id, unk_id):
        self.texts = df["text"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.word_to_id = word_to_id
        self.unk_id = unk_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenize(self.texts[idx])
        ids = [self.word_to_id.get(t, self.unk_id) for t in tokens]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float32)

def collate_batch(batch):
    # batch: list of (tensor_ids, label)
    ids_list, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in ids_list], dtype=torch.long)
    if len(ids_list) == 0:
        return torch.empty(0, dtype=torch.long), torch.empty(0)
    padded = pad_sequence(ids_list, batch_first=True, padding_value=pad_id)  # (batch, max_len)
    labels = torch.tensor(labels, dtype=torch.float32)
    return padded, lengths, labels

In [53]:
train_ds = SentimentDataset(train_df, word_to_id, unk_id)
val_ds = SentimentDataset(val_df, word_to_id, unk_id)
test_ds = SentimentDataset(test_df, word_to_id, unk_id)

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

## Train CNN

In [54]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for x, lengths, y in dataloader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            total_correct += (preds == y.long()).sum().item()
            total += x.size(0)
    avg_loss = total_loss / total
    acc = total_correct / total
    return avg_loss, acc

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    total = 0
    for x, lengths, y in tqdm(dataloader, desc="train"):
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)
    return total_loss / total

In [55]:
def select_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")


device = select_device()

In [56]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, embed_dim, pretrained_weights, filter_sizes=[2,3,4], num_filters=100, dropout=0.5, freeze_embeddings=False):
        super().__init__()
        # Embedding: initialize from pretrained Word2Vec-like vectors
        emb_tensor = torch.from_numpy(pretrained_weights)     # shape (vocab_size, embed_dim)
        self.embedding = nn.Embedding.from_pretrained(emb_tensor, freeze=freeze_embeddings, padding_idx=pad_id)

        # Conv1d expects input (batch, channels, seq_len) where channels = embed_dim
        self.filter_sizes = filter_sizes
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)  # single logit for binary classification

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)          # (batch, seq_len, embed_dim)
        emb = emb.transpose(1, 2)        # (batch, embed_dim, seq_len)
        conv_outputs = []
        for conv in self.convs:
            c = conv(emb)                # (batch, num_filters, seq_len - kernel_size + 1)
            c = F.relu(c)
            # Global max pooling over time dimension
            c = F.max_pool1d(c, kernel_size=c.size(2)).squeeze(2)  # (batch, num_filters)
            conv_outputs.append(c)
        cat = torch.cat(conv_outputs, dim=1)  # (batch, num_filters * len(filter_sizes))
        dropped = self.dropout(cat)
        logit = self.fc(dropped).squeeze(1)   # (batch,)
        return logit   # raw logits; use BCEWithLogitsLoss


In [59]:
model = TextCNN(
    embed_dim=V_in_extended.shape[1],
    pretrained_weights=V_in_extended,
    filter_sizes=[2, 3],
    num_filters=64,
    dropout=0.6
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1, verbose=True)

KeyboardInterrupt: 

In [39]:
best_val_acc = 0.0
history = {"train_loss": [], "val_loss": [], "val_acc": []}
ckpt_dir = "checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

num_epochs = 20
for epoch in range(1, num_epochs+1):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, device)
    scheduler.step(val_acc)  # LR scheduler keyed on val_acc

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    # checkpoint if improved by val_acc
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        ckpt_path = os.path.join(ckpt_dir, f"best_model_epoch{epoch}_acc{val_acc:.4f}.pt")
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc,
        }, ckpt_path)
        print("Saved new best checkpoint:", ckpt_path)



train: 100%|██████████| 134/134 [00:01<00:00, 104.08it/s]


Epoch 1: train_loss=0.6896, val_loss=0.6727, val_acc=0.5844
Saved new best checkpoint: checkpoints/best_model_epoch1_acc0.5844.pt



train: 100%|██████████| 134/134 [00:01<00:00, 130.33it/s]


Epoch 2: train_loss=0.6659, val_loss=0.6646, val_acc=0.6144
Saved new best checkpoint: checkpoints/best_model_epoch2_acc0.6144.pt



train: 100%|██████████| 134/134 [00:01<00:00, 126.23it/s]


...

Epoch 28: train_loss=0.0930, val_loss=0.6666, val_acc=0.7261



train: 100%|██████████| 134/134 [00:01<00:00, 129.21it/s]


Epoch 29: train_loss=0.0964, val_loss=0.6667, val_acc=0.7261



train: 100%|██████████| 134/134 [00:01<00:00, 110.28it/s]


Epoch 30: train_loss=0.0928, val_loss=0.6667, val_acc=0.7251
